# LFW·SurvFace 통합 Quick/Full 실험 실행기

이 노트북은 데이터셋별 legacy 노트북을 삭제하거나 서로 실행하지 않습니다. 동일한 `research/` Python 단계 함수를 한 곳에서 계획하고 순차 호출하는 상위 실행기입니다.

- `quick`: 실제 데이터와 실제 모델을 사용하되 LFW 10%, SurvFace 2%의 identity-aware·role-preserving 표본을 사용합니다.
- `full`: 전체 데이터 100%를 사용합니다.
- 모델은 `arc`, `ada`, `mag` 중 하나를 선택하고, 해당 profile과 로컬 가중치 경로를 명시합니다.
- `DATASET_IDS`에 한 개 또는 두 데이터셋을 선택하면 같은 모델·checkpoint로 순차 실행합니다. 세 모델 비교는 `MODEL_NAME`을 바꾸어 각각 독립 run으로 수행합니다.
- 완료 run마다 PCA direct/reconstruction 및 PQ reconstruction/ADC 파생 평가를 만들고, SurvFace에서는 선택적으로 faithfulness를 계산한 뒤 공통 보고서를 생성합니다.
- 실제 장시간 실험은 사용자가 직접 `EXECUTE=True`와 실행 확인 값을 설정해야 시작됩니다.
- 현재 공통 dispatcher는 Step 4 Grad-CAM workflow와 exhaustive PQ ADC를 지원합니다. IVF-PQ와 완전한 calibration 100/500/1,000 행렬은 별도 검증 범위입니다.


## 1. 사용자가 조절하는 변수

`DATASET_IDS`, `RUN_TIER`, `MODEL_NAME`을 선택합니다. quick 표본 비율은 `QUICK_DATA_FRACTIONS`에서 데이터셋별로 조절하며, 선택값과 기본값 차이는 실행 설정에 기록됩니다. `full`은 이 사전과 관계없이 항상 100%입니다.

`MODEL_PROFILE_BY_NAME`과 `MODEL_WEIGHT_PATHS`는 반드시 같은 학습 데이터·아키텍처의 조합이어야 합니다. `START_NEW_RUN=True`는 동일 plan의 완료 run이 이미 있는데도 독립 재실험을 의도할 때만 사용합니다.


In [ ]:
from __future__ import annotations

DATASET_IDS = ("survface",)  # 한 개만 실행하려면 ("survface",)
DATASET_ID = DATASET_IDS[0]  # 기존 단일-dataset 계약 호환용 별칭/
RUN_TIER = "full"       # "quick" 또는 "full"
QUICK_DATA_FRACTIONS = {
    "lfw": 1.0,
    "survface": 1.0,
}
SEED = 42

MODEL_NAME = "arc"  # "arc", "ada", "mag" 중 하나
MODEL_PROFILE_BY_NAME = {
    "arc": "arcface_ms1mv3_r100",
    "ada": "adaface_ms1mv3_r100",
    "mag": "magface_ms1mv2_iresnet100",
}
MODEL_WEIGHT_PATHS = {
    "arc": "models/arcface/ms1mv3_r100_backbone.pth",
    "ada": "models/adaface/adaface_ir101_ms1mv3.ckpt",
    "mag": "models/magface/magface_epoch_00025.pth",
}
# AdaFace MS1MV2 대안: profile을 adaface_ms1mv2_r100으로 바꾸고
# MODEL_WEIGHT_PATHS["ada"] = "models/adaface/adaface_ir101_ms1mv2.ckpt"
# 로 설정합니다. 현재 그 bridge profile은 Grad-CAM 실행이 비활성입니다.
MODEL_SMOKE_DEVICE = "cuda"
ARTIFACT_STORAGE_MODE = "results_only"  # "results_only": 정량 결과/정규화 heatmap만, "full": 모든 중간 배열·join CSV 보존


EXECUTE = True  # 실제 장시간 실험(pipeline)을 실행할지 여부 (True 시 실행)
ACKNOWLEDGE_LOCAL_EXECUTION = True  # 안전 장치: 로컬 환경에서의 실제 실행 승인 플래그
START_NEW_RUN = False  # 완료/실패한 동일 조건 run을 재사용·재개; 독립 재실험 때만 True
COMPLETED_RUN_OVERRIDES = {
    # source snapshot만 달라진 검증 완료 run을 명시적으로 재사용합니다.
    # 모델/프로토콜/데이터 범위가 다르면 fail-closed로 거부됩니다.
    "lfw": (
        "runs/lfw_20260803/"
        "20260803-R001-a4479011_step4_lfw_arcface-7972a704552df378345f"
    ),
}
RUN_SEARCH_SPACE_REFRESH = True  # 사후 처리(파생 평가) 과정에서 검색 공간 갱신 여부
RUN_SURVFACE_FAITHFULNESS = True  # SurvFace 데이터셋에 대한 Grad-CAM 기반 faithfulness 평가 여부
RUN_FINAL_REPORT = True  # 실험 완료 후 데이터셋/모델 간 교차 검증 최종 보고서 생성 여부
WRITE_FINAL_REPORT = True  # 생성된 최종 보고서 결과를 디스크에 저장/덮어쓰기 할지 여부



## 2. 프로젝트와 공통 runner 로드

현재 작업 디렉터리의 상위에서 저장소 루트를 찾습니다. 아래 셀은 실험을 시작하거나 가중치를 로드하지 않습니다.


In [4]:
from pathlib import Path
from pprint import pprint
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.pipeline_runner import (
    FULL_DATA_FRACTION,
    build_common_experiment_plan,
    inspect_common_experiment_plan,
    prepare_common_model_checkpoint,
    reuse_completed_run_for_plan,
    run_common_step4_experiment,
)
from research.runtime import ProgressReporter
from scripts.run_integrated_postprocessing import (
    postprocess_completed_run,
    run_cross_dataset_report_notebook,
    verify_cross_model_faithfulness,
)

if not DATASET_IDS or not set(DATASET_IDS).issubset({"lfw", "survface"}):
    raise ValueError(f"지원하지 않는 DATASET_IDS: {DATASET_IDS!r}")
if len(set(DATASET_IDS)) != len(DATASET_IDS):
    raise ValueError(f"DATASET_IDS에 중복이 있습니다: {DATASET_IDS!r}")

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"MODEL_NAME={MODEL_NAME}")
print(f"DATASET_IDS={DATASET_IDS}")
print(f"QUICK_DATA_FRACTIONS={QUICK_DATA_FRACTIONS}")
print(f"FULL_DATA_FRACTION={FULL_DATA_FRACTION}")


PROJECT_ROOT=C:\ronbun
MODEL_NAME=arc
DATASET_IDS=('lfw', 'survface')
QUICK_DATA_FRACTIONS={'lfw': 1.0, 'survface': 1.0}
FULL_DATA_FRACTION=1.0


## 3. 선택 모델과 가중치 고정

선택한 별칭에서 profile과 가중치 경로를 가져와 checkpoint SHA-256, 전처리, target layer를 하나의 `model_uid`로 등록합니다. 등록 정보가 같으면 재사용합니다. 기존 smoke 검증 결과가 있으면 재사용하고, 없으면 최대 8장으로 forward/target-layer smoke test를 수행합니다. 전체 데이터 실험은 시작하지 않습니다.


In [5]:
if MODEL_NAME not in MODEL_PROFILE_BY_NAME or MODEL_NAME not in MODEL_WEIGHT_PATHS:
    raise ValueError("MODEL_NAME은 'arc', 'ada', 'mag' 중 하나여야 합니다.")

MODEL_PROFILE = MODEL_PROFILE_BY_NAME[MODEL_NAME]
MODEL_WEIGHT_PATH = PROJECT_ROOT / MODEL_WEIGHT_PATHS[MODEL_NAME]

MODEL_PREPARATION = prepare_common_model_checkpoint(
    project_root=PROJECT_ROOT,
    model_name=MODEL_NAME,
    model_profile=MODEL_PROFILE,
    checkpoint_path=MODEL_WEIGHT_PATH,
    run_smoke_validation=True,
    smoke_device=MODEL_SMOKE_DEVICE,
    seed=SEED,
)
pprint(MODEL_PREPARATION.as_dict(), sort_dicts=False)


{'model_name': 'arc',
 'model_profile': 'arcface_ms1mv3_r100',
 'family': 'arcface',
 'checkpoint_path': 'C:\\ronbun\\models\\arcface\\ms1mv3_r100_backbone.pth',
 'checkpoint_sha256': 'a566a62357f0c55b679d9ff2f022a294486568be0c00665d39029d0e46a8109b',
 'model_uid': 'arcface-7972a704552df378345f',
 'model_spec_path': 'C:\\ronbun\\runs\\step2\\model_registry\\arcface-7972a704552df378345f.json',
 'smoke_validation_status': 'reused_validated',
 'smoke_validation_path': 'C:\\ronbun\\runs\\step2\\model_validation\\arcface-7972a704552df378345f\\smoke_summary.json'}


## 4. 결정적 실행 plan 생성

manifest를 읽어 실제 선택 예정 행 수와 role/split 분포를 계산합니다. 같은 manifest hash·seed·tier라면 같은 identity 집합을 선택합니다. 이 셀도 DB나 run을 변경하지 않습니다.


In [6]:
PLANS = {
    dataset_id: build_common_experiment_plan(
        project_root=PROJECT_ROOT,
        dataset_id=dataset_id,
        run_tier=RUN_TIER,
        seed=SEED,
        model_name=MODEL_NAME,
        model_profile=MODEL_PREPARATION.model_profile,
        model_uid=MODEL_PREPARATION.model_uid,
        model_checkpoint_path=MODEL_PREPARATION.checkpoint_path,
        quick_data_fractions=QUICK_DATA_FRACTIONS,
        artifact_storage_mode=ARTIFACT_STORAGE_MODE,
    )
    for dataset_id in DATASET_IDS
}
PLAN = PLANS[DATASET_ID]  # 기존 단일-plan 점검 코드와의 호환용 별칭
for dataset_id, plan in PLANS.items():
    print(f"\n[{dataset_id}] plan")
    pprint(plan.as_dict(), sort_dicts=False)


KeyError: 's'

## 5. 로컬 preflight

등록 checkpoint, CUDA, ONNX Runtime provider, canonical aligned/landmark bundle, 로컬 Git source 상태를 읽기 전용으로 검사합니다. GitHub나 원격 CI는 사용하지 않습니다. quick은 dirty source를 허용하되 commit과 diff hash를 plan에 고정하고, full은 clean local commit을 요구합니다. `ready_to_execute_pipeline=False`이면 아래 실행 셀의 오류에 표시되는 실패 항목을 먼저 해결합니다.


In [ ]:
PREFLIGHTS = {
    dataset_id: inspect_common_experiment_plan(plan)
    for dataset_id, plan in PLANS.items()
}
PREFLIGHT = PREFLIGHTS[DATASET_ID]  # 기존 단일-plan 점검 코드와의 호환용 별칭
for dataset_id, preflight in PREFLIGHTS.items():
    print(f"\n[{dataset_id}] preflight")
    pprint(preflight, sort_dicts=False)


## 6. 사용자 승인 후 순차 실행 또는 재개

`EXECUTE=True`, `ACKNOWLEDGE_LOCAL_EXECUTION=True`일 때만 실제 실험을 시작합니다. 실행 전에 위 plan과 preflight에서 데이터셋·fraction·model_uid·checkpoint 경로를 확인하십시오.

완료된 phase는 건너뛰고, 실패하거나 아직 실행하지 않은 phase부터 이어갑니다. 장시간 loop 로그는 약 10% 경계에서만 출력됩니다. 같은 plan의 완료 run이 있으면 자동으로 새 run을 만들지 않습니다.


In [ ]:
EXECUTION_RESULTS = {}
POSTPROCESS_RESULTS = {}
FINAL_REPORT_RESULT = {"status": "not_started"}

if EXECUTE:
    if ACKNOWLEDGE_LOCAL_EXECUTION is not True:
        raise RuntimeError(
            "실제 실행 전 ACKNOWLEDGE_LOCAL_EXECUTION=True가 필요합니다."
        )
    for dataset_id, preflight in PREFLIGHTS.items():
        if preflight["ready_to_execute_pipeline"]:
            continue
        CHECKS = preflight["readiness"]["checks"]
        FAILED_CHECKS = {
            key: CHECKS.get(key)
            for key in (
                "git_policy_satisfied",
                "source_snapshot_matches",
                "cuda_available",
                "required_onnx_provider_available",
                "model_spec_verified",
            )
            if CHECKS.get(key) is not True
        }
        raise RuntimeError(f"preflight 실패: {FAILED_CHECKS}")

    for dataset_id, plan in PLANS.items():
        progress = ProgressReporter(
            f"{dataset_id}/{RUN_TIER}/{MODEL_NAME}",
            heartbeat_seconds=None,
            milestone_percent=10,
        )
        override = COMPLETED_RUN_OVERRIDES.get(dataset_id)
        execution = (
            reuse_completed_run_for_plan(plan, PROJECT_ROOT / override)
            if override
            else run_common_step4_experiment(
                plan,
                execution_acknowledged=True,
                start_new_run=START_NEW_RUN,
                progress=progress.callback(
                    key_prefix=f"{dataset_id}:{RUN_TIER}:"
                ),
            )
        )
        EXECUTION_RESULTS[dataset_id] = execution
        if execution.get("status") not in {"completed", "already_completed"}:
            raise RuntimeError(f"{dataset_id}: 완료 run을 확보하지 못했습니다: {execution}")

    # 모든 dataset run을 먼저 완료한 뒤 파생 artifact를 만든다. 첫 dataset의
    # results가 다음 dataset의 source preflight에 영향을 주지 않게 한다.
    for dataset_id, execution in EXECUTION_RESULTS.items():
        POSTPROCESS_RESULTS[dataset_id] = postprocess_completed_run(
            execution["run_dir"],
            refresh_search_spaces=RUN_SEARCH_SPACE_REFRESH,
            derive_survface_faithfulness=RUN_SURVFACE_FAITHFULNESS,
        )

    if RUN_FINAL_REPORT:
        survface_execution = EXECUTION_RESULTS.get("survface")
        cross_model = verify_cross_model_faithfulness(
            PROJECT_ROOT,
            expected_survface_run_id=(
                survface_execution["run_id"] if survface_execution else None
            ),
            expected_model_uid=(
                MODEL_PREPARATION.model_uid if survface_execution else None
            ),
        )
        FINAL_REPORT_RESULT = run_cross_dataset_report_notebook(
            PROJECT_ROOT,
            model_name=MODEL_NAME,
            selected_runs={
                dataset_id: result["run_dir"]
                for dataset_id, result in EXECUTION_RESULTS.items()
            },
            include_survface_faithfulness=(
                cross_model["status"] == "completed"
            ),
            write_outputs=WRITE_FINAL_REPORT,
            overwrite_outputs=True,
        )
        FINAL_REPORT_RESULT["cross_model_faithfulness"] = cross_model
else:
    EXECUTION_RESULTS = {
        dataset_id: {
            "status": "not_started",
            "reason": "EXECUTE=False; plan과 preflight만 수행했습니다.",
        }
        for dataset_id in DATASET_IDS
    }

EXECUTION_RESULT = EXECUTION_RESULTS[DATASET_ID]  # 기존 단일-result 호환용 별칭
INTEGRATED_RESULT = {
    "execution": EXECUTION_RESULTS,
    "postprocessing": POSTPROCESS_RESULTS,
    "final_report": FINAL_REPORT_RESULT,
}
pprint(INTEGRATED_RESULT, sort_dicts=False)


## 7. 결과 해석 경계

- `quick` 결과는 코드·DB·artifact 흐름과 경향 확인용이며 논문 최종 수치가 아닙니다.
- `full`은 전체 표본이라는 조건만 충족합니다. 동일 commit에서 LFW/SurvFace를 모두 재실행하고, 모델 UID·전처리·평가 계약 parity를 확인해야 직접 비교할 수 있습니다.
- ArcFace·AdaFace·MagFace 비교 단위는 선택한 pretrained checkpoint입니다. loss 함수 자체의 인과적 우월성 비교로 해석하지 않습니다.
- PQ reconstruction cosine과 exhaustive ADC는 별도 search mode로 보고합니다. ADC 결과를 reconstruction cosine 또는 IVF-PQ latency로 해석하지 않습니다.
- SurvFace origin calibration transfer가 복구되기 전에는 DIR·FPIR·threshold crossing을 목표 FPIR 성능으로 승격하지 않습니다.
- Grad-CAM faithfulness는 high/low/random 동일 조건의 파생 artifact가 있을 때만 공통 보고서에 포함하며, 3-model 통합 artifact가 없으면 보고서 생성 자체는 계속합니다.
- 데이터셋별 세부 단계 디버깅은 기존 `notebooks/lfw` 또는 `notebooks/survface` 순차 노트북을 사용합니다.
